# MLB Edge Finder — Exploration

Interactive walkthrough of the pipeline stages.

In [1]:
import logging
from datetime import date

from mlb_edge_finder import config

from pybaseball import cache
cache.enable()

config.setup_logging(level=logging.INFO)

## 1. Fetch Odds

In [2]:
from mlb_edge_finder import odds_ingestion

game_date = date.today()
odds_df = odds_ingestion.fetch_odds(game_date, force=True, debug=True)
odds_df.head()

2026-04-28 19:44:16,036 | INFO | mlb_edge_finder.odds_ingestion | Excluded 6 already-started game(s) — live in-game odds are not used
2026-04-28 19:44:16,041 | INFO | mlb_edge_finder.odds_ingestion | Raw API response: 15 game(s) returned
2026-04-28 19:44:16,041 | INFO | mlb_edge_finder.odds_ingestion |   game_id=abd0114b226ed04c6bd4daac824f5241  home=Chicago White Sox  away=Los Angeles Angels  commence_time=2026-04-29T17:11:00Z  local_date=2026-04-29
2026-04-28 19:44:16,042 | INFO | mlb_edge_finder.odds_ingestion |   game_id=de142860d352b18879f2382e67ab3e2e  home=Cleveland Guardians  away=Tampa Bay Rays  commence_time=2026-04-29T17:11:00Z  local_date=2026-04-29
2026-04-28 19:44:16,043 | INFO | mlb_edge_finder.odds_ingestion |   game_id=3bf2df80de60f0bf82343a646a188fb2  home=Minnesota Twins  away=Seattle Mariners  commence_time=2026-04-29T17:41:00Z  local_date=2026-04-29
2026-04-28 19:44:16,043 | INFO | mlb_edge_finder.odds_ingestion |   game_id=af3e6454c0293bcd38866af00fb22cc1  home=Te

,game_id,home_team,away_team,home_odds_american,away_odds_american,commence_time


## 2. Fetch Stats

In [7]:
from mlb_edge_finder import stats_ingestion

stats_df = stats_ingestion.fetch_stats(game_date)
stats_df.head()

2026-04-28 20:21:03,874 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs attempt 1/3 failed: Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403 — retrying in 2s
2026-04-28 20:21:05,959 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs attempt 2/3 failed: Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403 — retrying in 4s
2026-04-28 20:21:10,022 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs failed after 3 attempts (Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403) — falling back to MLB Stats API
2026-04-28 20:21:10,450 | INFO | mlb_edge_finder.stats_ingestion | Wrote 30 rows to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/raw/stats_2026-04-28.csv (source: mlb_api)


,team_abbr,bat_avg,obp,slg,ops,runs_per_game,era,whip,k_per_9,bb_per_9,fip_computed,data_source
0,LAD,.275,.352,.461,.813,5.466667,3.25,1.10,8.84,3.07,3.457456,mlb_api
1,ATL,.274,.340,.455,.795,5.700000,3.09,1.13,8.43,3.19,3.747015,mlb_api
2,CHC,.261,.352,.423,.775,5.266667,4.04,1.21,8.18,3.08,4.039734,mlb_api
3,HOU,.260,.344,.440,.784,5.133333,5.96,1.64,9.44,5.51,5.080295,mlb_api
4,TB,.253,.330,.385,.715,4.689655,4.13,1.23,7.99,3.58,4.410054,mlb_api


## 3. Build Features

In [8]:
from mlb_edge_finder import features

features_df = features.build_features(game_date)
features_df.head()
features_df[features_df.isnull().any(axis=1)]
features_df[['home_team', 'away_team', 'home_bat_avg', 'away_bat_avg', 'home_era', 'away_era']]

2026-04-28 20:21:19,216 | INFO | mlb_edge_finder.features | Wrote 0 rows to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/processed/features_2026-04-28.csv


,home_team,away_team,home_bat_avg,away_bat_avg,home_era,away_era


## 4a. Historical Ingestion

Fetch completed regular season results for each training season via `statsapi`.

In [5]:
from mlb_edge_finder import historical_ingestion

# Fetch (or load cached) results for a single season
hist_2024 = historical_ingestion.fetch_historical(2024)
print(f"{len(hist_2024)} games")
hist_2024.head()

2428 games


,game_date,home_name,away_name,home_score,away_score,home_win
0,2024-03-20,San Diego Padres,Los Angeles Dodgers,2,5,0
1,2024-03-21,Los Angeles Dodgers,San Diego Padres,11,15,0
2,2024-03-28,Baltimore Orioles,Los Angeles Angels,11,3,1
3,2024-03-28,Cincinnati Reds,Washington Nationals,8,2,1
4,2024-03-28,San Diego Padres,San Francisco Giants,6,4,1


In [9]:
# Concatenate all training seasons (2023, 2024, 2025)
all_hist = historical_ingestion.fetch_all_historical()
print(f"{len(all_hist)} total games")
all_hist.groupby(all_hist['game_date'].str[:4])['home_win'].agg(['count', 'mean'])

2026-04-28 20:21:23,832 | INFO | mlb_edge_finder.historical_ingestion | fetch_all_historical: 7273 total games across seasons [2023, 2024, 2025]


7273 total games


,count,mean
game_date,,
2023,2418,0.520678
2024,2428,0.521005
2025,2427,0.543881


## 4b. Training Data

Join end-of-season team stats (one snapshot per year) to each game row to produce the model training set.

In [10]:
from mlb_edge_finder import training_data

seasons = [2023, 2024, 2025]
training_df = training_data.build_training_set(seasons)
print(f"{len(training_df)} rows, {training_df.shape[1]} columns")
training_df.head()

2026-04-28 20:39:00,246 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs attempt 1/3 failed: Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403 — retrying in 2s
2026-04-28 20:39:02,324 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs attempt 2/3 failed: Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403 — retrying in 4s
2026-04-28 20:39:06,395 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs failed after 3 attempts (Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403) — falling back to MLB Stats API
2026-04-28 20:39:06,910 | INFO | mlb_edge_finder.stats_ingestion | Wrote 30 rows to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/raw/stats_2023-09-28.csv (source: mlb_api)
2026-04-28 20:39:06,985 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs attempt 1/3 failed: Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Receive

6787 rows, 29 columns


,game_date,home_name,away_name,home_score,away_score,home_win,home_abbr,away_abbr,home_bat_avg,home_obp,...,away_obp,away_slg,away_ops,away_runs_per_game,away_era,away_whip,away_k_per_9,away_bb_per_9,away_fip_computed,season
0,2023-03-30,Washington Nationals,Atlanta Braves,2,7,0,WSH,ATL,.254,.314,...,.344,.501,.845,5.845679,4.14,1.30,9.48,3.34,3.845139,2023
1,2023-03-30,New York Yankees,San Francisco Giants,5,0,1,NYY,SF,.227,.304,...,.312,.383,.695,4.160494,4.02,1.25,8.53,2.53,3.665967,2023
2,2023-03-30,Boston Red Sox,Baltimore Orioles,9,10,0,BOS,BAL,.258,.324,...,.321,.421,.742,4.981481,3.89,1.24,8.86,2.93,3.740421,2023
3,2023-03-30,Chicago Cubs,Milwaukee Brewers,4,0,1,CHC,MIL,.254,.330,...,.319,.385,.704,4.493827,3.71,1.19,8.89,3.07,3.983680,2023
4,2023-03-30,Tampa Bay Rays,Detroit Tigers,4,0,1,TB,DET,.260,.331,...,.305,.382,.687,4.080247,4.24,1.25,8.57,2.97,3.920404,2023


In [11]:
# Class balance and missing-value check
print("home_win distribution:")
print(training_df['home_win'].value_counts())
print()
nulls = training_df.isnull().sum()
print("Null counts:", nulls[nulls > 0].to_dict() or "none")

home_win distribution:
home_win
1    3593
0    3194
Name: count, dtype: int64

Null counts: none


## 4c. Model Training

Train an XGBoost classifier on the training set, evaluate it, and persist the model and metrics.

In [ ]:
from datetime import date
from mlb_edge_finder import model

clf, X_test, y_test = model.train(training_df)
print(f"Test set size: {len(X_test)} games")
print(f"Features used: {list(X_test.columns)}")

In [ ]:
baseline_clf, _, _ = model.train_baseline(training_df)
print("Baseline (logistic regression) trained.")

In [ ]:
import pandas as pd

xgb_metrics = model.evaluate(clf, X_test, y_test)
lr_metrics = model.evaluate(baseline_clf, X_test, y_test)

comparison = pd.DataFrame(
    {"XGBoost": xgb_metrics, "LogisticRegression": lr_metrics},
    index=xgb_metrics.keys(),
)
print(comparison.to_string())

model.save_model(clf, xgb_metrics, date.today())

In [ ]:
loaded_clf = model.load_model(date.today())
print("Model reloaded successfully.")
print(f"Sample predictions: {loaded_clf.predict(X_test[:3])}")

## 5. Find Edges

Run inference on today's features and flag games where the model finds positive expected value.

In [ ]:
from mlb_edge_finder import edge_finder

edges = edge_finder.find_edges(features_df, clf, game_date)
print(f"{len(edges)} edge(s) found for {game_date}")
edges

### 5b. Full Pipeline (end-to-end)

`pipeline.run()` drives all five stages — odds fetch, stats fetch, feature build, model load, edge find — in one call.

In [ ]:
from mlb_edge_finder import pipeline

# Runs end-to-end for today: fetch odds, stats, build features, load model, find edges
pipeline_edges = pipeline.run(game_date)
pipeline_edges